# Part A: Multimodal document retrieval, Group 22

Executable record of the real lab run (BITS lab, NVIDIA A100, qwen3:8b through Ollama). Run all cells in the lab. Cells call the repository scripts; nothing here is mocked. The ingest takes minutes, so this notebook reads the saved index and the saved results, then runs the integrated demonstration (E10) live.

In [ ]:
import csv, json, os, subprocess, sys
from pathlib import Path

ROOT = Path.home() / 'multimodal-rag-group22'
PY = str(Path.home() / 'venv_rag' / 'bin' / 'python')
if not os.path.exists(PY):
    PY = sys.executable

def run(cmd, **kw):
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, **kw)
    print(p.stdout[-3000:] + p.stderr[-800:])
    return p.returncode

print('repo:', ROOT, '| python:', PY)

## 1. Environment (GPU, model, packages)

In [ ]:
run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'])
run([str(Path.home() / 'ollama012' / 'bin' / 'ollama'), 'list'])
run(['git', 'log', '--oneline', '-1'])

## 2. Corpus and ingestion (E1)

In [ ]:
print(json.dumps(json.load(open(ROOT / 'results' / 'ingest_summary.json')), indent=1)[:1500])
print(json.dumps(json.load(open(ROOT / 'results' / 'e1_inventory.json')), indent=1)[:1200])

## 3. Integrated demonstration (E10): rate limit, input guardrail, memory, retrieval, prompt, LLM, output guardrail

In [ ]:
rc = run([PY, 'scripts/e10_demo.py', '--embedder', 'st', '--llm', 'ollama', '--model', 'qwen3:8b', '--results', 'results/e10'])
print((ROOT / 'results' / 'e10' / 'e10_demo.md').read_text()[:6000])

## 4. Saved results of the final run (30 golden questions)

In [ ]:
fin = ROOT / 'results' / 'final_30'
for name in ['e3_retrieval.csv', 'e7_ablation.csv', 'e2_chunking.csv']:
    print('==', name)
    print((fin / name).read_text())
print('== e5_summary (all)'); print(json.dumps(json.load(open(fin / 'e5_summary.json'))['all']))
print('== e6_benchmark'); print(json.dumps({k: v for k, v in json.load(open(fin / 'e6_benchmark.json')).items() if k != 'unstable_queries'}, indent=1))
print('== e5_auto_eval_summary'); print((fin / 'e5_auto_eval_summary.json').read_text())

## 5. Memory demonstration (E4)

In [ ]:
print((ROOT / 'results' / 'final_30' / 'e4_transcript.md').read_text())

## 6. Offline tests (fake LLM and hash embedder; they check the code, they are not results)

In [ ]:
run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], env={**os.environ, 'PYTHONPATH': str(ROOT / 'src')})